# Flujo MX real: pacs.008 → CSV Banco → Cliente → Conciliación

Este notebook ejecuta todo el flujo en **un solo lugar**.

**Requisitos**: `pandas` (incluido en el `environment.yml`).

In [1]:
# Rutas base (ajusta si mueves la carpeta)
BASE = r"/mnt/data/pacs008_flow_mx_real"
sample_xml = f"{BASE}/sample/pacs008_sample_mx_real.xml"
bank_csv_out = f"{BASE}/out/reconciliation_to_client_mx.csv"
client_orders = f"{BASE}/client/client_orders_mx.csv"
client_results = f"{BASE}/out/client_reconciliation_results_mx.csv"

print(sample_xml)
print(bank_csv_out)
print(client_orders)
print(client_results)

/mnt/data/pacs008_flow_mx_real/sample/pacs008_sample_mx_real.xml
/mnt/data/pacs008_flow_mx_real/out/reconciliation_to_client_mx.csv
/mnt/data/pacs008_flow_mx_real/client/client_orders_mx.csv
/mnt/data/pacs008_flow_mx_real/out/client_reconciliation_results_mx.csv


In [ ]:
import pandas as pd
import numpy as np

# Ruta relativa a Documents
adult_df = pd.read_csv('Documents/adult.csv')

## 1) Banco: convertir pacs.008 a CSV de conciliación

In [2]:
from xml.etree import ElementTree as ET
from datetime import datetime
import csv

NS = {"ns": "urn:iso:std:iso:20022:tech:xsd:pacs.008.001.08"}

def _text(e): 
    return (e.text or "").strip() if e is not None else ""

def find_first(elem, paths):
    for p in paths:
        x = elem.find(p, NS)
        if x is not None and _text(x):
            return _text(x)
    return ""

def pacs008_to_rows(xml_path, credit_ts):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    grp_hdr = root.find(".//ns:GrpHdr", NS)
    msg_id = _text(grp_hdr.find("ns:MsgId", NS)) if grp_hdr is not None else ""
    cre_dt_tm = _text(grp_hdr.find("ns:CreDtTm", NS)) if grp_hdr is not None else ""

    rows = []
    for i, tx in enumerate(root.findAll(".//ns:CdtTrfTxInf", NS) if hasattr(root, "findAll") else root.findall(".//ns:CdtTrfTxInf", NS), start=1):
        instr_id = _text(tx.find("ns:PmtId/ns:InstrId", NS))
        end2end  = _text(tx.find("ns:PmtId/ns:EndToEndId", NS))
        tx_id    = _text(tx.find("ns:PmtId/ns:TxId", NS))
        amt_el   = tx.find("ns:IntrBkSttlmAmt", NS)
        amount   = _text(amt_el) if amt_el is not None else ""
        ccy      = amt_el.get("Ccy") if amt_el is not None else ""
        chrg_br  = _text(tx.find("ns:ChrgBr", NS))
        uetr     = _text(tx.find(".//ns:UETR", NS)) if tx.find(".//ns:UETR", NS) is not None else ""

        dbtr_name    = _text(tx.find("ns:Dbtr/ns:Nm", NS))
        dbtr_account = find_first(tx, ["ns:DbtrAcct/ns:Id/ns:Othr/ns:Id", "ns:DbtrAcct/ns:Id/ns:IBAN"])
        cdtr_name    = _text(tx.find("ns:Cdtr/ns:Nm", NS))
        cdtr_account = find_first(tx, ["ns:CdtrAcct/ns:Id/ns:Othr/ns:Id", "ns:CdtrAcct/ns:Id/ns:IBAN"])
        cdtr_bic     = find_first(tx, ["ns:CdtrAgt/ns:FinInstnId/ns:BICFI"])
        purp         = _text(tx.find("ns:Purp/ns:Cd", NS))
        remittance   = " | ".join([_text(u) for u in tx.findall("ns:RmtInf/ns:Ustrd", NS) if _text(u)])

        rows.append({ 
            "bank_msg_id": msg_id,
            "creation_datetime": cre_dt_tm,
            "transaction_seq": i,
            "instr_id": instr_id,
            "end_to_end_id": end2end,
            "tx_id": tx_id,
            "uetr": uetr,
            "amount": amount,
            "currency": ccy,
            "charges_bearer": chrg_br,
            "debtor_name": dbtr_name,
            "debtor_account": dbtr_account,
            "creditor_name": cdtr_name,
            "creditor_account": cdtr_account,
            "creditor_bic": cdtr_bic,
            "purpose_code": purp,
            "remittance_information": remittance,
            "credit_posting_ts": credit_ts
        })
    return rows

# Ejecutar conversión
rows = pacs008_to_rows(sample_xml, datetime.utcnow().isoformat())
cols = ["bank_msg_id","creation_datetime","transaction_seq","instr_id","end_to_end_id","tx_id","uetr",
        "amount","currency","charges_bearer","debtor_name","debtor_account","creditor_name",
        "creditor_account","creditor_bic","purpose_code","remittance_information","credit_posting_ts"]

import pandas as pd
df_bank = pd.DataFrame(rows, columns=cols)
df_bank.to_csv(bank_csv_out, index=False)
df_bank

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/pacs008_flow_mx_real/sample/pacs008_sample_mx_real.xml'

## 2) Cliente: cargar órdenes internas (CSV demo)

In [ ]:
import pandas as pd
df_orders = pd.read_csv(client_orders, dtype=str)
if "amount" in df_orders.columns:
    df_orders["amount"] = df_orders["amount"].astype(float)
df_orders

## 3) Cliente: reconciliar CSV del banco vs órdenes

In [ ]:
from difflib import SequenceMatcher
import pandas as pd

def norm(s):
    try:
        return str(s if s is not None else "").strip().lower()
    except Exception:
        return ""

def similar(a,b):
    a,b = norm(a), norm(b)
    if not a or not b: return 0.0
    return SequenceMatcher(None, a, b).ratio()

def score(bank_row, order_row, amount_tol=0.01):
    score = 0.0; reasons = []
    if norm(bank_row.get("end_to_end_id","")) and norm(bank_row.get("end_to_end_id","")) == norm(order_row.get("end_to_end_id","")):
        score += 0.55; reasons.append("EndToEndId exacto")
    if norm(bank_row.get("uetr","")) and norm(bank_row.get("uetr","")) == norm(order_row.get("uetr","")):
        score += 0.25; reasons.append("UETR exacto")
    if norm(bank_row.get("instr_id","")) and norm(bank_row.get("instr_id","")) == norm(order_row.get("instr_id","")):
        score += 0.10; reasons.append("InstrId exacto")
    if norm(bank_row.get("tx_id","")) and norm(bank_row.get("tx_id","")) == norm(order_row.get("tx_id","")):
        score += 0.10; reasons.append("TxId exacto")
    try:
        if abs(float(bank_row.get("amount",0.0)) - float(order_row.get("amount",0.0))) <= amount_tol:
            score += 0.30; reasons.append("Monto exacto")
    except Exception:
        pass
    if norm(bank_row.get("currency","")) == norm(order_row.get("currency","")):
        score += 0.20; reasons.append("Divisa coincide")

    order_acct = order_row.get("creditor_clabe") or order_row.get("creditor_iban") or ""
    if norm(bank_row.get("creditor_account","")) and norm(bank_row.get("creditor_account","")) == norm(order_acct):
        score += 0.25; reasons.append("Cuenta beneficiario (CLABE/IBAN) exacta")

    if norm(bank_row.get("creditor_bic","")) and norm(bank_row.get("creditor_bic","")) == norm(order_row.get("creditor_bic","")):
        score += 0.15; reasons.append("BIC beneficiario exacto")

    ref_hits = []
    for token in [order_row.get("invoice_ref",""), order_row.get("po_ref",""), order_row.get("order_id","")]:
        t = norm(token)
        if t and t in norm(bank_row.get("remittance_information","")):
            ref_hits.append(token)
    if ref_hits:
        score += 0.25; reasons.append(f"Referencia en remittance: {', '.join(ref_hits)}")

    name_sim = max(similar(bank_row.get("creditor_name",""), order_row.get("creditor_name","")),
                   similar(bank_row.get("remittance_information",""), order_row.get("creditor_name","")))
    if name_sim >= 0.85:
        score += 0.10; reasons.append(f"Nombre beneficiario ~ {name_sim:.2f}")

    return min(score, 1.0), reasons

def reconcile_df(df_bank, df_orders, auto_threshold=0.75, suggest_threshold=0.55):
    rows_out = []
    for _, tx in df_bank.iterrows():
        best = None; best_score = -1.0; best_reasons = []
        for _, od in df_orders.iterrows():
            s, r = score(tx, od)
            if s > best_score:
                best_score, best, best_reasons = s, od, r
        status = "NEEDS_REVIEW"
        if best_score >= auto_threshold: status = "AUTO_MATCH"
        elif best_score >= suggest_threshold: status = "SUGGESTION"
        rows_out.append({
            "bank_msg_id": tx.get("bank_msg_id",""),
            "creation_datetime": tx.get("creation_datetime",""),
            "transaction_seq": tx.get("transaction_seq",""),
            "instr_id": tx.get("instr_id",""),
            "end_to_end_id": tx.get("end_to_end_id",""),
            "tx_id": tx.get("tx_id",""),
            "uetr": tx.get("uetr",""),
            "amount": tx.get("amount",""),
            "currency": tx.get("currency",""),
            "creditor_name": tx.get("creditor_name",""),
            "creditor_account": tx.get("creditor_account",""),
            "creditor_bic": tx.get("creditor_bic",""),
            "remittance_information": tx.get("remittance_information",""),
            "credit_posting_ts": tx.get("credit_posting_ts",""),
            "status": status,
            "confidence": round(best_score,3),
            "matched_order_id": (best.get("order_id","") if best is not None else ""),
            "matched_invoice_ref": (best.get("invoice_ref","") if best is not None else ""),
            "matched_po_ref": (best.get("po_ref","") if best is not None else ""),
            "signals": "; ".join(best_reasons)
        })
    return pd.DataFrame(rows_out)

df_res = reconcile_df(df_bank, df_orders)
df_res.to_csv(client_results, index=False)
df_res